# STATE K562 Benchmark v3

**Step 1:** Run Cell 1 below (installs packages, then auto-restarts)

**Step 2:** After restart, click **Run all** — Cell 1 will skip the install and everything proceeds automatically

In [ ]:
# Cell 1: Install and restart (only runs pip once)
import importlib
try:
    importlib.import_module('state')
    import torch, scanpy, numpy as np
    print(f"All packages ready. numpy={np.__version__}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: No GPU! Set Runtime > Change runtime type > T4")
    print("Proceeding to next cell...")
except ImportError:
    print("First run: installing packages...")
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'arc-state', 'scanpy', 'anndata', 'scipy',
                           'huggingface_hub'])
    print("\n" + "="*50)
    print("INSTALLED. Restarting runtime now...")
    print("After restart, click Run All.")
    print("="*50)
    import os
    os.kill(os.getpid(), 9)

In [ ]:
# Cell 2: Download data and model
import os
import urllib.request
from huggingface_hub import snapshot_download

DATA_PATH = "ReplogleWeissman2022_K562_essential.h5ad"
MODEL_DIR = "ST-HVG-Parse"

if not os.path.exists(DATA_PATH):
    print("Downloading K562 data (1.5 GB)...")
    url = "https://zenodo.org/records/7041849/files/ReplogleWeissman2022_K562_essential.h5ad?download=1"
    urllib.request.urlretrieve(url, DATA_PATH)
    print(f"Downloaded: {os.path.getsize(DATA_PATH)/1e9:.1f} GB")
else:
    print(f"Data present: {os.path.getsize(DATA_PATH)/1e9:.1f} GB")

if not os.path.exists(MODEL_DIR):
    print("Downloading ST-HVG-Parse model...")
    snapshot_download("arcinstitute/ST-HVG-Parse", local_dir=MODEL_DIR)
    print("Model downloaded.")
else:
    print("Model present.")

print("\nModel files:")
for root, dirs, files in os.walk(MODEL_DIR):
    for f in files:
        path = os.path.join(root, f)
        print(f"  {path} ({os.path.getsize(path)/1e6:.0f} MB)")

In [ ]:
# Cell 3: Load and preprocess data
import scanpy as sc
import numpy as np

print("Loading K562 data...")
adata = sc.read_h5ad(DATA_PATH)
print(f"Shape: {adata.shape}")

pert_col = None
for col in ["gene", "perturbation", "guide_id", "condition", "target_gene"]:
    if col in adata.obs.columns:
        pert_col = col
        break

ctrl_label = None
for label in ["non-targeting", "control", "ctrl", "non_targeting"]:
    if label in adata.obs[pert_col].values:
        ctrl_label = label
        break

print(f"Pert column: '{pert_col}', Control: '{ctrl_label}'")
print(f"Targets: {len([p for p in adata.obs[pert_col].unique() if p != ctrl_label])}")

# Preprocess
PROC_PATH = "k562_for_state.h5ad"
adata_proc = adata.copy()
sc.pp.normalize_total(adata_proc, target_sum=1e4)
sc.pp.log1p(adata_proc)
sc.pp.highly_variable_genes(adata_proc, n_top_genes=2000)

hvg_idx = np.where(adata_proc.var.highly_variable)[0]
X_hvg = adata_proc.X[:, hvg_idx]
if hasattr(X_hvg, 'toarray'):
    X_hvg = X_hvg.toarray()
adata_proc.obsm["X_hvg"] = X_hvg
adata_proc.write(PROC_PATH)
print(f"Preprocessed: {X_hvg.shape}, saved to {PROC_PATH}")

In [ ]:
# Cell 4: Run STATE inference (the long step)
import subprocess
import time

OUTPUT_PATH = "state_predictions.h5ad"

# Find checkpoint file
ckpt_path = None
for root, dirs, files in os.walk(MODEL_DIR):
    for f in sorted(files):
        if f.endswith((".ckpt", ".safetensors", ".pt", ".bin")):
            ckpt_path = os.path.join(root, f)
            break
    if ckpt_path:
        break

print(f"Checkpoint: {ckpt_path}")

cmd = [
    "state", "tx", "infer",
    "--model-dir", MODEL_DIR,
    "--adata", PROC_PATH,
    "--pert-col", pert_col,
    "--embed-key", "X_hvg",
    "--output", OUTPUT_PATH,
]
if ckpt_path:
    cmd.extend(["--checkpoint", ckpt_path])

print(f"Command: {' '.join(cmd)}\n")
print("Running inference (may take 30 min - 3 hours)...\n")

t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
elapsed = (time.time() - t0) / 60
print(f"\nDone in {elapsed:.1f} min (exit code: {proc.returncode})")

In [ ]:
# Cell 5: Score predictions
from scipy import stats
import csv

print("Loading predictions...")
adata_pred = sc.read_h5ad(OUTPUT_PATH)
adata_obs = sc.read_h5ad(DATA_PATH)
sc.pp.normalize_total(adata_obs, target_sum=1e4)
sc.pp.log1p(adata_obs)

print(f"Pred: {adata_pred.shape}, Obs: {adata_obs.shape}")
print(f"Pred obs cols: {list(adata_pred.obs.columns[:10])}")
print(f"Pred layers: {list(adata_pred.layers.keys())}")
print(f"Pred obsm: {list(adata_pred.obsm.keys())}")

# Align gene space
shared = sorted(set(adata_pred.var_names) & set(adata_obs.var_names))
print(f"Shared genes: {len(shared)}")

if len(shared) == 0:
    sc.pp.highly_variable_genes(adata_obs, n_top_genes=2000)
    hvg_names = adata_obs.var_names[adata_obs.var.highly_variable]
    if adata_pred.shape[1] == len(hvg_names):
        print(f"HVG dimension match: {len(hvg_names)}")
        adata_pred.var_names = hvg_names
        shared = list(hvg_names)
        adata_obs = adata_obs[:, hvg_names]

if len(shared) > 0:
    adata_pred = adata_pred[:, shared]
    adata_obs = adata_obs[:, shared]

# Control mean
ctrl_X = adata_obs[adata_obs.obs[pert_col] == ctrl_label].X
if hasattr(ctrl_X, 'toarray'):
    ctrl_X = ctrl_X.toarray()
ctrl_mean = ctrl_X.mean(axis=0)

# Find pert column in predictions
pred_pert_col = None
for col in [pert_col, "perturbation", "gene", "target_gene"]:
    if col in adata_pred.obs.columns:
        pred_pert_col = col
        break

# Score each perturbation
obs_perts = [p for p in adata_obs.obs[pert_col].unique() if p != ctrl_label]
print(f"\nScoring {len(obs_perts)} genes...")

scores = []
failed = 0
for i, gene in enumerate(obs_perts):
    if (i + 1) % 200 == 0:
        print(f"  [{i+1}/{len(obs_perts)}] scored={len(scores)} failed={failed}")
    try:
        obs_mask = adata_obs.obs[pert_col] == gene
        n_obs = obs_mask.sum()
        if n_obs < 5:
            failed += 1
            continue
        obs_X = adata_obs[obs_mask].X
        if hasattr(obs_X, 'toarray'):
            obs_X = obs_X.toarray()
        obs_delta = np.array(obs_X.mean(axis=0) - ctrl_mean).flatten()

        pred_mask = adata_pred.obs[pred_pert_col] == gene
        if pred_mask.sum() == 0:
            failed += 1
            continue
        pred_X = adata_pred[pred_mask].X
        if hasattr(pred_X, 'toarray'):
            pred_X = pred_X.toarray()
        pred_delta = np.array(pred_X.mean(axis=0) - ctrl_mean).flatten()

        if np.std(obs_delta) > 1e-10 and np.std(pred_delta) > 1e-10:
            r, p = stats.pearsonr(obs_delta, pred_delta)
        else:
            r, p = 0.0, 1.0

        scores.append({
            "gene": gene,
            "pearson_correlation": round(float(r), 6),
            "pearson_pvalue": round(float(p), 10),
            "n_perturbed_cells": int(n_obs),
            "method": "STATE_ST-HVG-Parse",
        })
    except:
        failed += 1

print(f"\nDone: {len(scores)} scored, {failed} failed")
accs = [s['pearson_correlation'] for s in scores]
if accs:
    print(f"Mean r: {np.mean(accs):.4f}, Median: {np.median(accs):.4f}")

In [ ]:
# Cell 6: Save, plot, and download
import matplotlib.pyplot as plt

OUTPUT_TSV = "state_k562_per_gene_scores.tsv"
with open(OUTPUT_TSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "gene", "pearson_correlation", "pearson_pvalue",
        "n_perturbed_cells", "method"
    ], delimiter="\t")
    writer.writeheader()
    writer.writerows(scores)
print(f"Saved: {OUTPUT_TSV} ({len(scores)} genes)")

plt.figure(figsize=(10, 4))
plt.hist(accs, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(np.mean(accs), color='red', linestyle='--',
            label=f'Mean={np.mean(accs):.3f}')
plt.xlabel('Pearson r')
plt.ylabel('Count')
plt.title('STATE ST-HVG-Parse: Per-gene accuracy on K562')
plt.legend()
plt.tight_layout()
plt.show()

from google.colab import files
files.download(OUTPUT_TSV)
print("\nPlace the downloaded TSV in progframe/results/")
print("Then run: python merge_state_results.py")